# Ask Anand AI — RAGfolio Notebook

This notebook builds a recruiter-facing RAG assistant for Anand Chamoli using:

- Knowledge base documents
- SentenceTransformer embeddings
- ChromaDB vector database
- Semantic retrieval
- Local extractive answers
- Optional GPT-powered answer generation if an OpenAI API key is available

**Recommended folder name:** `KnowledgeBase`  
**Vector database folder:** `vector_db`


## 1. Install Required Libraries

Run this cell once. If Jupyter asks to restart the kernel after installation, restart and continue from the next cell.


In [1]:
!pip install -U chromadb sentence-transformers langchain-text-splitters pypdf python-docx openai


  Attempting uninstall: pypdf
    Found existing installation: pypdf 6.14.0
    Uninstalling pypdf-6.14.0:
      Successfully uninstalled pypdf-6.14.0


## 2. Import Libraries and Set Paths


In [2]:
import os
import re
import shutil
from pathlib import Path

from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from chromadb import PersistentClient

# Change this only if your folder name is different
KNOWLEDGE_BASE_PATH = Path("KnowledgeBase")
VECTOR_DB_PATH = "vector_db"
COLLECTION_NAME = "ask_anand"

print("Current working directory:", Path.cwd())
print("Knowledge base folder exists:", KNOWLEDGE_BASE_PATH.exists())


Current working directory: /Users/anandchamoli/KnowledgeBase
Knowledge base folder exists: False


## 3. Load Knowledge Base Files

This loader intentionally skips `.rtf`, `.DS_Store`, images, and other files that should not be indexed.  
Allowed file types: `.md`, `.txt`, `.pdf`, `.docx`


In [3]:
from pypdf import PdfReader
from docx import Document

ALLOWED_EXTENSIONS = {".md", ".txt", ".pdf", ".docx"}
SKIP_EXTENSIONS = {".rtf", ".png", ".jpg", ".jpeg", ".DS_Store"}


def clean_text(text: str) -> str:
    """Clean whitespace and remove common leftover formatting artifacts."""
    text = text.replace(" 
", "
")
    text = text.replace(" ", " ")
    text = re.sub(r"\\[a-zA-Z0-9]+", " ", text)  # defensive cleanup for accidental RTF commands
    text = re.sub(r"
{3,}", "

", text)
    text = re.sub(r"[ 	]{2,}", " ", text)
    return text.strip()


def read_txt_md(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="ignore")


def read_pdf(path: Path) -> str:
    reader = PdfReader(str(path))
    pages = []
    for page in reader.pages:
        pages.append(page.extract_text() or "")
    return "
".join(pages)


def read_docx(path: Path) -> str:
    doc = Document(str(path))
    paragraphs = [p.text for p in doc.paragraphs if p.text.strip()]
    return "
".join(paragraphs)


def load_knowledge_base(base_path: Path):
    documents = []
    skipped = []

    for path in base_path.rglob("*"):
        if path.is_dir():
            continue
        if path.name.startswith("."):
            skipped.append((str(path), "hidden/system file"))
            continue

        ext = path.suffix.lower()
        if ext not in ALLOWED_EXTENSIONS:
            skipped.append((str(path), f"unsupported extension: {ext}"))
            continue

        try:
            if ext in {".md", ".txt"}:
                text = read_txt_md(path)
            elif ext == ".pdf":
                text = read_pdf(path)
            elif ext == ".docx":
                text = read_docx(path)
            else:
                text = ""

            text = clean_text(text)
            if text:
                documents.append({
                    "source": str(path),
                    "text": text
                })
        except Exception as e:
            skipped.append((str(path), f"read error: {e}"))

    return documents, skipped


documents, skipped_files = load_knowledge_base(KNOWLEDGE_BASE_PATH)

print("Documents loaded:", len(documents))
print("Files skipped:", len(skipped_files))
print("
Loaded files:")
for doc in documents:
    print("-", doc["source"])

print("
Skipped files:")
for item in skipped_files[:30]:
    print("-", item)


SyntaxError: unterminated string literal (detected at line 10) (1289420669.py, line 10)

## 4. Quick Data Quality Check

This confirms that `.rtf` formatting codes are not entering the vector database.


In [ ]:
for i, doc in enumerate(documents[:5], 1):
    print(f"
--- Document {i}: {doc['source']} ---")
    print(doc["text"][:500])

# Check for RTF contamination
rtf_contaminated = [d["source"] for d in documents if "\\f1" in d["text"] or "\\pard" in d["text"] or "{\\rtf" in d["text"]]
print("
RTF contaminated files found:", len(rtf_contaminated))
for f in rtf_contaminated:
    print("-", f)


## 5. Split Documents into Chunks

Chunks are smaller text sections that can be embedded and retrieved based on user questions.


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120
)

chunks = []
metadatas = []

for doc in documents:
    split_texts = splitter.split_text(doc["text"])
    for chunk_id, chunk in enumerate(split_texts):
        chunks.append(chunk)
        metadatas.append({
            "source": doc["source"],
            "chunk_id": chunk_id
        })

print("Total chunks created:", len(chunks))
print("
Sample chunk:")
print(chunks[0][:700])


## 6. Load Local Embedding Model

This uses free local embeddings, so no OpenAI API key is required for retrieval.


In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

test_embedding = embedding_model.encode("Does Anand have GCP experience?")
print("Embedding dimension:", len(test_embedding))


## 7. Create ChromaDB Vector Database

This converts all chunks into embeddings and stores them in a local ChromaDB database.


In [ ]:
# Optional: delete old vector database to rebuild cleanly
if Path(VECTOR_DB_PATH).exists():
    shutil.rmtree(VECTOR_DB_PATH)

client = PersistentClient(path=VECTOR_DB_PATH)
collection = client.create_collection(name=COLLECTION_NAME)

embeddings = embedding_model.encode(chunks, show_progress_bar=True)

collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print("Vector database created successfully.")
print("Total chunks stored:", collection.count())


## 8. Retrieval Function

This searches the vector database and returns the most relevant knowledge-base chunks.


In [ ]:
def retrieve_context(query: str, n_results: int = 5):
    query_embedding = embedding_model.encode(query)

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=n_results
    )

    retrieved = []
    for doc, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        retrieved.append({
            "text": doc,
            "source": meta.get("source", "Unknown"),
            "distance": distance
        })

    return retrieved


def print_retrieval_results(query: str, n_results: int = 5):
    retrieved = retrieve_context(query, n_results=n_results)
    print("QUERY:", query)
    for i, item in enumerate(retrieved, 1):
        print(f"
--- Result {i} ---")
        print("Source:", item["source"])
        print("Distance:", round(item["distance"], 4))
        print(item["text"][:900])

print_retrieval_results("Does Anand have GCP experience?", n_results=3)


## 9. Local Recruiter-Ready Answer Function

This version does not require GPT. It retrieves relevant evidence and presents it in a cleaner format for testing.


In [ ]:
def ask_anand_local(query: str, n_results: int = 5):
    retrieved = retrieve_context(query, n_results=n_results)

    answer_lines = []
    answer_lines.append(f"Question: {query}")
    answer_lines.append("
Answer based on Anand's verified knowledge base:
")

    for i, item in enumerate(retrieved, 1):
        answer_lines.append(f"Evidence {i} | Source: {Path(item['source']).name}")
        answer_lines.append(item["text"])
        answer_lines.append("
" + "-" * 80 + "
")

    return "
".join(answer_lines)

print(ask_anand_local("Does Anand have GCP experience?", n_results=3))


## 10. Optional GPT Answer Function

Use this only when you have a valid OpenAI API key.  
Do not hardcode your key in GitHub. Use an environment variable instead:

```bash
export OPENAI_API_KEY="your_key_here"
```


In [ ]:
def ask_anand_gpt(query: str, n_results: int = 6, model: str = "gpt-4o-mini"):
    import os
    from openai import OpenAI

    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        return "OpenAI API key not found. Use ask_anand_local() or set OPENAI_API_KEY."

    retrieved = retrieve_context(query, n_results=n_results)
    context = "

".join([
        f"Source: {Path(item['source']).name}
{item['text']}"
        for item in retrieved
    ])

    prompt = f"""
You are Ask Anand AI, a professional recruiter-facing portfolio assistant for Anand Chamoli.

Answer the user's question using only the verified context below.
Use a concise, professional tone suitable for recruiters, HR managers, VPs, and business leaders.
If the answer is not available in the context, say:
"I do not have verified information about that in Anand's knowledge base."

Question:
{query}

Verified Context:
{context}

Answer:
"""

    client = OpenAI(api_key=api_key)
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a concise, professional career portfolio assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

# Example after setting OPENAI_API_KEY:
# print(ask_anand_gpt("Why should we hire Anand?"))


## 11. Recruiter Demo Questions

Run these questions to test the RAG assistant before converting it into Streamlit.


In [ ]:
demo_questions = [
    "Why should we hire Anand?",
    "Does Anand have GCP experience?",
    "Tell me about CARE Direct 24x7.",
    "Is Anand suitable for a Business Analyst role?",
    "What analytics projects has Anand completed?",
    "How does Anand's tractor industry experience help in analytics roles?",
    "What can Anand deliver in the first 90 days?",
    "Can Anand help with customer retention strategy?",
]

for q in demo_questions:
    print("=" * 100)
    print(ask_anand_local(q, n_results=3))
    print("
")


## 12. JD Gap Analyzer — Retrieval Context Starter

This is the first version of the job-description analyzer. It retrieves Anand's strongest matching evidence for a pasted JD.


In [ ]:
def analyze_jd_local(job_description: str, n_results: int = 8):
    query = f"""
    Compare Anand Chamoli's profile with this job description.
    Identify matching skills, strengths, gaps, and role fit.

    Job Description:
    {job_description}
    """

    retrieved = retrieve_context(query, n_results=n_results)

    output = []
    output.append("JD Match Evidence from Anand's Knowledge Base")
    output.append("=" * 70)

    for i, item in enumerate(retrieved, 1):
        output.append(f"
Evidence {i} | Source: {Path(item['source']).name}")
        output.append(item["text"][:1200])
        output.append("-" * 70)

    return "
".join(output)

sample_jd = "Business Analyst role requiring SQL, Python, dashboarding, stakeholder management, sales analytics, customer analytics, and business reporting."
print(analyze_jd_local(sample_jd, n_results=5))


## 13. Save a Simple Project Summary for GitHub


In [ ]:
project_summary = """
# Ask Anand AI - RAGfolio

Ask Anand AI is a Retrieval-Augmented Generation style professional portfolio assistant built using Python, ChromaDB, SentenceTransformer embeddings, and a custom career knowledge base.

## Features
- Recruiter question answering
- Skills evidence retrieval
- Project exploration
- CARE Direct 24x7 case study retrieval
- JD match evidence retrieval
- Optional GPT-powered response generation

## Core Technologies
- Python
- ChromaDB
- SentenceTransformers
- LangChain Text Splitters
- OpenAI API (optional)
- Jupyter Notebook

## Knowledge Base
The assistant uses Anand Chamoli's resume, LinkedIn profile, analytics projects, CARE Direct 24x7 case study, skills matrix, target roles, domain expertise, and recruiter Q&A.
"""

Path("Ask_Anand_AI_Project_Summary.md").write_text(project_summary.strip(), encoding="utf-8")
print("Project summary saved as Ask_Anand_AI_Project_Summary.md")


# Next Step

After this notebook works well, convert it into:

- `ingest.py`
- `rag_engine.py`
- `app.py`
- `jd_analyzer.py`

Then build the Streamlit web app for recruiters.


In [4]:
python ingest.py

SyntaxError: invalid syntax (1575829381.py, line 1)

ModuleNotFoundError: No module named 'rag_engine'